In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import time

def build_model(config):
    model = models.Sequential()
    model.add(layers.Flatten(input_shape=(64, 64, 1)))
    for neurons in config:
        model.add(layers.Dense(neurons, activation='relu'))
    model.add(layers.Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

configs = [
    [16],                # Model 1
    [128],               # Model 2
    [64, 32, 16],        # Model 3
    [128, 64, 32, 16]    # Model 4
]

results = []
for i, cfg in enumerate(configs):
    model = build_model(cfg)
    start_time = time.time()
    # Training (represented logic)
    # history = model.fit(train_images, train_labels, epochs=10, verbose=0)
    end_time = time.time()
    results.append({'model': i+1, 'time': end_time - start_time})

/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
pip install tensorflow opencv-python numpy patool scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.6/88.6 kB 3.1 MB/s eta 0:00:00


In [ ]:
import os
import time
import cv2
import numpy as np
import patoolib
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, models

# 1. EXTRAER EL DATASET
archivo_rar = "Dataset imagenes IA - Tonos grises (1).rar"
path_extraccion = "dataset_grises"

# Check if the rar file exists
if not os.path.exists(archivo_rar):
    print(f"Error: El archivo RAR '{archivo_rar}' no se encuentra en el directorio actual.")
    print("Por favor, asegúrate de subir el archivo 'Dataset imagenes IA - Tonos grises (1).rar' a la sesión de Colab.")
    # Raise an exception to stop execution and prevent the ValueError
    raise FileNotFoundError(f"'{archivo_rar}' not found. Please upload the dataset.")

if not os.path.exists(path_extraccion):
    os.makedirs(path_extraccion)
    print(f"Extrayendo '{archivo_rar}' a '{path_extraccion}'...")
    try:
        patoolib.extract_archive(archivo_rar, outdir=path_extraccion)
        print("Extracción completada.")
    except Exception as e:
        print(f"Error al extraer el archivo RAR: {e}")
        # Re-raise the exception to indicate a critical failure
        raise
else:
    print(f"El directorio de extracción '{path_extraccion}' ya existe. Saltando la extracción.")

# 2. CARGAR Y PREPROCESAR IMÁGENES
def cargar_datos(directorio_base):
    imagenes = []
    etiquetas = []
    # Carpetas esperadas (ajustar nombres si varían en tu rar)
    categorias = {"Imagenes obscuras": 0, "Imagenes claras": 1}

    print(f"Buscando imágenes en: {directorio_base}")
    found_any_image = False
    for carpeta, etiqueta in categorias.items():
        ruta_carpeta = os.path.join(directorio_base, carpeta)
        if not os.path.exists(ruta_carpeta):
            print(f"Advertencia: La carpeta esperada '{ruta_carpeta}' no existe. Saltando esta categoría.")
            continue
        else:
            print(f"Procesando carpeta: {ruta_carpeta}")

        for archivo in os.listdir(ruta_carpeta):
            ruta_img = os.path.join(ruta_carpeta, archivo)
            img = cv2.imread(ruta_img, cv2.IMREAD_GRAYSCALE)
            if img is not None:
                img = cv2.resize(img, (64, 64)) # Redimensionar
                imagenes.append(img)
                etiquetas.append(etiqueta)
                found_any_image = True
            # else:
            #     print(f"Advertencia: No se pudo cargar la imagen o no es una imagen válida: {ruta_img}") # Too verbose

    if not found_any_image:
        print("Advertencia: No se encontraron imágenes en las carpetas esperadas.")
    return np.array(imagenes) / 255.0, np.array(etiquetas)

# Cargar los datos
X, y = cargar_datos(path_extraccion)
print(f"Total de imágenes cargadas: {len(X)}") # Added for diagnosis

if len(X) == 0:
    print("Error: No se cargaron imágenes. Asegúrate de que el archivo RAR contenga las carpetas 'Imagenes obscuras' y 'Imagenes claras' con imágenes dentro, y que se hayan extraído correctamente.")
    # Raise an exception if no data to process, to prevent the ValueError from train_test_split
    raise ValueError("No images loaded, cannot proceed with train_test_split.")

X = X.reshape(-1, 64, 64, 1) # Ajustar forma para la red
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. DEFINIR LAS 4 CONFIGURACIONES
configuraciones = {
    "Modelo A (Simple)": [16],
    "Modelo B (Equilibrado)": [64, 32],
    "Modelo C (Profundo)": [128, 64, 32],
    "Modelo D (Ancho)": [512]
}

resultados = {}

# 4. ENTRENAMIENTO Y EVALUACIÓN
print(f"--- Iniciando pruebas con {len(X)} imágenes ---
")

for nombre, capas in configuraciones.items():
    model = models.Sequential()
    model.add(layers.Flatten(input_shape=(64, 64, 1)))
    for neuronas in capas:
        model.add(layers.Dense(neuronas, activation='relu'))
    model.add(layers.Dense(1, activation='sigmoid'))

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    inicio = time.time()
    # Entrenamos por 10 épocas
    historia = model.fit(X_train, y_train, epochs=10, batch_size=8, verbose=0)
    tiempo_total = time.time() - inicio

    loss, acc = model.evaluate(X_test, y_test, verbose=0)
    resultados[nombre] = {'accuracy': acc, 'tiempo': tiempo_total}

    print(f"{nombre}: Precisión = {acc:.4f} | Tiempo = {tiempo_total:.2f}s")

# 5. SELECCIÓN DEL MEJOR
mejor_modelo = max(resultados, key=lambda k: resultados[k]['accuracy'])
print(f"\n--- EL GANADOR ---")
print(f"El mejor modelo es el '{mejor_modelo}' con {resultados[mejor_modelo]['accuracy']*100:.2f}% de precisión.")

SyntaxError: unterminated f-string literal (detected at line 89) (ipython-input-3606577552.py, line 89)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import cv2
import numpy as np
import time
import pandas as pd
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, models

# 1. CONFIGURACIÓN DE RUTAS Y CARGA
# Ajustamos la ruta según donde unrar extrae los archivos
ruta_raiz = "/content/dataset"
# A veces unrar crea una subcarpeta con el mismo nombre del rar, verificamos ambas
carpetas_objetivo = ["Imagenes obscuras", "Imagenes claras"]

def cargar_dataset():
    X = []
    y = []

    # Buscamos las carpetas en la raíz o un nivel abajo
    for etiqueta_nombre, etiqueta_val in [("Imagenes obscuras", 0), ("Imagenes claras", 1)]:
        ruta_final = ""
        for root, dirs, files in os.walk(ruta_raiz):
            if etiqueta_nombre in dirs:
                ruta_final = os.path.join(root, etiqueta_nombre)
                break

        if ruta_final:
            print(f"Cargando desde: {ruta_final}")
            for img_name in os.listdir(ruta_final):
                img_path = os.path.join(ruta_final, img_name)
                # Leer en escala de grises
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                if img is not None:
                    img = cv2.resize(img, (64, 64)) # Redimensionar para uniformidad
                    X.append(img)
                    y.append(etiqueta_val)
        else:
            print(f"⚠️ No se encontró la carpeta: {etiqueta_nombre}")

    return np.array(X), np.array(y)

# Cargar y normalizar
X, y = cargar_dataset()
if len(X) == 0:
    print("❌ Error: No se cargaron imágenes. Verifica las rutas.")
else:
    X = X / 255.0  # Normalizar píxeles de 0-255 a 0-1
    X = X.reshape(-1, 64, 64, 1) # Ajustar forma para la red neuronal

    # Dividir en entrenamiento y prueba (80% / 20%)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # 2. DEFINIR LAS 4 COMBINACIONES DE NEURONAS Y CAPAS
    # Probaremos diferentes profundidades y anchos
    configuraciones = [
        {"nombre": "Config 1: Muy Simple", "capas": [8]},
        {"nombre": "Config 2: Una Capa Ancha", "capas": [128]},
        {"nombre": "Config 3: Dos Capas (Medio)", "capas": [64, 32]},
        {"nombre": "Config 4: Tres Capas (Profundo)", "capas": [128, 64, 32]}
    ]



    resultados = []

    print(f"\n--- Iniciando Entrenamiento de {len(X)} imágenes ---\n")

    for conf in configuraciones:
        model = models.Sequential()
        model.add(layers.Flatten(input_shape=(64, 64, 1)))

        # Agregar capas según la configuración
        for neuronas in conf["capas"]:
            model.add(layers.Dense(neuronas, activation='relu'))

        # Capa de salida (Sigmoid para clasificación binaria)
        model.add(layers.Dense(1, activation='sigmoid'))

        model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

        # Medir tiempo de entrenamiento
        inicio = time.time()
        # Entrenamos por 20 épocas dado que el dataset es pequeño
        historia = model.fit(X_train, y_train, epochs=20, batch_size=4, verbose=0)
        fin = time.time()

        # Evaluación
        loss, acc = model.evaluate(X_test, y_test, verbose=0)
        tiempo_total = fin - inicio

        resultados.append({
            "Configuración": conf["nombre"],
            "Precisión (Accuracy)": f"{acc*100:.2f}%",
            "Tiempo (seg)": round(tiempo_total, 3),
            "Estructura": conf["capas"]
        })

    # 3. MOSTRAR RESULTADOS Y SELECCIONAR EL MEJOR
    df_resultados = pd.DataFrame(resultados)
    print(df_resultados.to_string(index=False))

    mejor = max(resultados, key=lambda x: float(x["Precisión (Accuracy)"].replace('%','')))
    print(f"\n🌟 EL MEJOR MODELO ES: {mejor['Configuración']}")
    print(f"Razón: Logró una precisión de {mejor['Precisión (Accuracy)']} en {mejor['Tiempo (seg)']} segundos.")

⚠️ No se encontró la carpeta: Imagenes obscuras
⚠️ No se encontró la carpeta: Imagenes claras
❌ Error: No se cargaron imágenes. Verifica las rutas.


In [1]:
import cv2
import numpy as np
import time
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split

# --- 1. CARGA EXPLÍCITA IMAGEN POR IMAGEN ---
def cargar_imagenes_manual():
    X = []
    y = []

    # Lista de rutas exactas (asumiendo que están en las carpetas que creaste al subir)
    # Si las subiste sueltas a /content/, ajusta la ruta.

    # Cargando las 11 Oscuras
    print("Iniciando lectura de imágenes obscuras...")
    for i in range(1, 12):
        ruta = f"/content/Imagenes obscuras/Imagen{i}.png"
        img = cv2.imread(ruta, cv2.IMREAD_GRAYSCALE)
        if img is not None:
            img = cv2.resize(img, (64, 64))
            X.append(img)
            y.append(0) # Etiqueta para Obscuras
            print(f"✅ Leída con éxito: {ruta}")
        else:
            print(f"❌ Error al leer: {ruta}")

    # Cargando las 12 Claras
    print("\nIniciando lectura de imágenes claras...")
    for i in range(1, 13):
        # Nota: Si tus claras siguen la numeración 12-23, cambia range(12, 24) e "Imagen{i}"
        ruta = f"/content/Imagenes claras/Imagen{i}.png"
        img = cv2.imread(ruta, cv2.IMREAD_GRAYSCALE)
        if img is not None:
            img = cv2.resize(img, (64, 64))
            X.append(img)
            y.append(1) # Etiqueta para Claras
            print(f"✅ Leída con éxito: {ruta}")
        else:
            print(f"❌ Error al leer: {ruta}")

    return np.array(X), np.array(y)

# Procesar datos
X_raw, y_raw = cargar_imagenes_manual()

if len(X_raw) == 0:
    print("\nNo se pudieron cargar imágenes. Revisa que los nombres y carpetas coincidan.")
else:
    # Normalización (esencial para que la red no tarde siglos en aprender)
    X = X_raw / 255.0
    X = X.reshape(-1, 64, 64, 1)

    # Dividir datos (80% entrenamiento, 20% prueba)
    X_train, X_test, y_train, y_test = train_test_split(X, y_raw, test_size=0.2, random_state=42)

    # --- 2. PRUEBA DE LAS 4 COMBINACIONES (Neuronas y Capas) ---
    configuraciones = [
        {"nombre": "Modelo A: 1 Capa (16 neuronas)", "capas": [16]},
        {"nombre": "Modelo B: 2 Capas (64, 32)", "capas": [64, 32]},
        {"nombre": "Modelo C: 3 Capas (128, 64, 32)", "capas": [128, 64, 32]},
        {"nombre": "Modelo D: 1 Capa Ancha (512)", "capas": [512]}
    ]

    print(f"\nTotal de imágenes para entrenamiento: {len(X_train)}")
    print("-" * 50)

    for conf in configuraciones:
        # Construir modelo dinámicamente
        model = models.Sequential()
        model.add(layers.Flatten(input_shape=(64, 64, 1)))
        for n in conf["capas"]:
            model.add(layers.Dense(n, activation='relu'))
        model.add(layers.Dense(1, activation='sigmoid'))

        model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

        # Medir tiempo
        inicio = time.time()
        # Entrenamos 25 épocas para que logre aprender con tan pocos datos
        model.fit(X_train, y_train, epochs=25, batch_size=4, verbose=0)
        tiempo = time.time() - inicio

        # Evaluar
        _, acc = model.evaluate(X_test, y_test, verbose=0)

        print(

SyntaxError: incomplete input (ipython-input-1174126469.py, line 88)